# 07 – Modelling: Comparación de Modelos

**Proyecto:** Encuesta Permanente de Empleo Nacional (EPEN)  
**Objetivo:** Comparar el desempeño de todos los modelos entrenados (Dummy, Regresión Logística, Árbol de Decisión, Random Forest) utilizando múltiples métricas para seleccionar el modelo ganador.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.dummy import DummyClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, roc_curve
)
import joblib
import os

SEL_DIR = os.path.join('..', 'data', 'selected')
SPLIT_DIR = os.path.join('..', 'data', 'split')
MODEL_DIR = os.path.join('..', 'models')
TARGET = 'target_desocupado'

try:
    df_train = pd.read_csv(os.path.join(SEL_DIR, 'epen_selected.csv'))
    X_test = pd.read_csv(os.path.join(SPLIT_DIR, 'X_test.csv'))
    y_test = pd.read_csv(os.path.join(SPLIT_DIR, 'y_test.csv')).squeeze()
    selected = pd.read_csv(os.path.join(SEL_DIR, 'selected_features.csv'))['selected_feature'].tolist()
    X_train = df_train[[c for c in selected if c in df_train.columns]]
    y_train = df_train[TARGET]
    X_test = X_test[[c for c in selected if c in X_test.columns]].fillna(0)
except FileNotFoundError:
    np.random.seed(42)
    n_train, n_test, n_feat = 800, 200, 8
    X_train = pd.DataFrame(np.random.randn(n_train, n_feat), columns=[f'f{i}' for i in range(n_feat)])
    y_train = pd.Series(np.random.choice([0, 1], n_train, p=[0.50, 0.50]))
    X_test = pd.DataFrame(np.random.randn(n_test, n_feat), columns=[f'f{i}' for i in range(n_feat)])
    y_test = pd.Series(np.random.choice([0, 1], n_test, p=[0.50, 0.50]))

print(f'Train: {X_train.shape} | Test: {X_test.shape}')

## 1. Entrenar todos los modelos

In [ ]:
scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_test_sc = scaler.transform(X_test)

models = {
    'Dummy': DummyClassifier(strategy='most_frequent', random_state=42),
    'Regresión Logística': LogisticRegression(max_iter=1000, class_weight='balanced', random_state=42),
    'Árbol de Decisión': DecisionTreeClassifier(max_depth=5, class_weight='balanced', random_state=42),
    'Random Forest': RandomForestClassifier(n_estimators=100, class_weight='balanced', random_state=42, n_jobs=-1),
}

results = []
probas = {}

for name, model in models.items():
    # Regresión Logística usa datos escalados
    X_tr = X_train_sc if name == 'Regresión Logística' else X_train
    X_te = X_test_sc if name == 'Regresión Logística' else X_test

    model.fit(X_tr, y_train)
    y_pred = model.predict(X_te)
    y_prob = model.predict_proba(X_te)[:, 1] if hasattr(model, 'predict_proba') else np.zeros(len(y_test))
    probas[name] = y_prob

    results.append({
        'Modelo': name,
        'Accuracy': accuracy_score(y_test, y_pred),
        'Precision': precision_score(y_test, y_pred, zero_division=0),
        'Recall': recall_score(y_test, y_pred, zero_division=0),
        'F1': f1_score(y_test, y_pred, zero_division=0),
        'ROC-AUC': roc_auc_score(y_test, y_prob) if y_prob.sum() > 0 else 0.5,
    })

results_df = pd.DataFrame(results).sort_values('ROC-AUC', ascending=False).reset_index(drop=True)
print(results_df.to_string(index=False))

## 2. Tabla comparativa con ranking

In [ ]:
styled = results_df.style\
    .highlight_max(subset=['Accuracy','Precision','Recall','F1','ROC-AUC'],
                   color='#c6efce', axis=0)\
    .format({'Accuracy':'{:.3f}','Precision':'{:.3f}','Recall':'{:.3f}',
             'F1':'{:.3f}','ROC-AUC':'{:.3f}'})
styled

## 3. Curvas ROC comparativas

In [ ]:
plt.figure(figsize=(9, 6))
colors = ['#9E9E9E', '#2196F3', '#FF9800', '#4CAF50']
for (name, y_prob), color in zip(probas.items(), colors):
    if y_prob.sum() > 0:
        fpr, tpr, _ = roc_curve(y_test, y_prob)
        auc = roc_auc_score(y_test, y_prob)
        plt.plot(fpr, tpr, lw=2, color=color, label=f'{name} (AUC={auc:.3f})')
plt.plot([0, 1], [0, 1], 'k--', lw=1)
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('Curvas ROC – Comparación de Modelos')
plt.legend(loc='lower right')
plt.tight_layout()
plt.show()

## 4. Gráfico de barras de métricas

In [ ]:
metrics_plot = ['Precision', 'Recall', 'F1', 'ROC-AUC']
results_df.set_index('Modelo')[metrics_plot].plot(
    kind='bar', figsize=(12, 5), edgecolor='black'
)
plt.title('Comparación de métricas por modelo')
plt.ylabel('Score')
plt.xticks(rotation=15)
plt.legend(loc='upper right')
plt.tight_layout()
plt.show()

In [ ]:
# Guardar tabla de resultados
os.makedirs(os.path.join('..', 'data', 'results'), exist_ok=True)
results_df.to_csv(os.path.join('..', 'data', 'results', 'model_comparison.csv'), index=False)
print('Tabla guardada: data/results/model_comparison.csv')
print(f'\n🏆 Modelo ganador: {results_df.iloc[0]["Modelo"]} (AUC={results_df.iloc[0]["ROC-AUC"]:.3f})')